In [1]:
# ============================================================
# PHASE 1.1 FINAL HANDCRAFTED PROTEIN BASELINE
# Project: T2D Gene/Protein Association Prediction
#
# Main experiment:
#   Feature set = AAC + Physicochemical descriptors
#   Models = Logistic Regression, SVM RBF, Random Forest, LightGBM,
#            Soft Voting, Stacking
#
# Ablation experiment:
#   Feature sets = Length only, AAC only, AAC + Physchem,
#                  AAC + DPC, Full
#   Models = Logistic Regression, Random Forest
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from itertools import product
import joblib
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", None)


# ============================================================
# SKLEARN IMPORTS
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)


# ============================================================
# LIGHTGBM IMPORT
# ============================================================

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
    print("LightGBM is available.")
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM is not installed. Using HistGradientBoostingClassifier fallback.")

LightGBM is available.


In [2]:
# ============================================================
# PATHS
# ============================================================

DATA_PATH = Path(r"D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_protein_classification_dataset_balanced_primary_only_v1.csv")
BASE_DIR = Path(r"D:\khoane\MAHE\Project\model\phase1_protein_baseline")

SPLIT_DIR = BASE_DIR / "splits"
FEATURE_DIR = BASE_DIR / "features"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

for folder in [SPLIT_DIR, FEATURE_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Base output directory:", BASE_DIR)

Base output directory: D:\khoane\MAHE\Project\model\phase1_protein_baseline


In [3]:
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

print("\nColumns:")
for col in df.columns:
    print("-", col)

Dataset shape: (1820, 30)


,gene_id,gene_symbol,gene_name,uniprot_accession,uniprot_entry_name,sequence,sequence_length_fasta,label,label_name,dataset_role,class_source,classification_dataset_version,association_score,positive_tier,label_source,label_rule,label_version,negative_sample_version,negative_sampling_rule,mapping_route,uniprot_to_ensg_status,representative_sequence_rule,length_bin,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
0,ENSG00000161618,ALDH16A1,ALDH16A1,Q8IZ83,A16A1_HUMAN,MAATRAGPRAREIFTSLEYGPVPESHACALAWLDTQDRCLGHYVNGKWLKPEHRNSVPCQDPITGENLASCLQAQAEDVAAAVEAARMAFKGWSAHPGVVRAQHLTRLAEVIQKHQRLLWTLESLVTGRAVREVRDGDVQLAQQLLHYHAIQASTQEEALAGWEPMGVIGLILPPTFSFLEMMWRICPALAVGCTVVALVPPASPAPLLLAQLAGELGPFPGILNVLSGPASLVPILASQPGIRKVAFCGAPEEGRALRRSLAGECAELGLALGTESLLLLTDTADVDSAVEGVVDAAWSDRGPGGLRLLIQESVWDEAMRRLQERMGRLRSGRGLDGAVDMGARGAAACDLVQRFVREAQSQGAQVFQAGDVPSERPFYPPTLVSNLPPASPCAQVEVPWPVVVASPFRTAKEALLVANGTPRGGSASVWSERLGQALELGYGLQVGTVWINAHGLRDPSVPTGGCKESGCSWHGGPDGLYEYLRPSGTPARLSCLSKNLNYDTFGLAVPSTLPAGPEIGPSPAPPYGLFVGGRFQAPGARSSRPIRDSSGNLHGYVAEGGAKDIRGAVEAAHQAFPGWAGQSPGARAALLWALAAALERRKSTLASRLERQGAELKAAEAEVELSARRLRAWGARVQAQGHTLQVAGLRGPVLRLREPLGVLAVVCPDEWPLLAFVSLLAPALAYGNTVVMVPSAACPLLALEVCQDMATVFPAGLANVVTGDRDHLTRCLALHQDVQAMWYFGSAQGSQFVEWASAGNLKPVWASRGCPRAWDQEAEGAGPELGLRVARTKALWLPMGD,802.0,0,T2D_negative,negative,Category_B_negative_clean_pool_lengthmatched_primary_coordinates,T2D_Protein_Classification_Balanced_PrimaryCoordinates_v1,NaN,NaN,NaN,NaN,NaN,Negative_Balanced_LengthMatched_PrimaryCoordinates_v1,Sampled from Negative Candidate Pool Clean v1 using 1:1 positive-negative balance with sequence-length-stratified sampling based on positive-set decile bins; non-primary patch/scaffold-only genes were excluded and replaced within the same length bins; random_state=42,NaN,one_uniprot_to_one_ensg,NaN,"(683.2, 852.6]",ENSG00000161618,ALDH16A1,protein_coding,19,49453146.0,49471052.0,1.0
1,ENSG00000224420,ADM5,ADM5,C9JUS6,ADM5_HUMAN,MTIHILILLLLLAFSAQGDLDTAARRGQHQVPQHRGHVCYLGVCRTHRLAEIIYWIRCLHQGALGEGQPRAPGPLQLWAPPVARGGSPARFPGFRPAARGLAQCPARWVTSGTARPLLGFSLPICMLELLLHISSPLTPAPETVFPSPSPGCD,153.0,0,T2D_negative,negative,Category_B_negative_clean_pool_lengthmatched_primary_coordinates,T2D_Protein_Classification_Balanced_PrimaryCoordinates_v1,NaN,NaN,NaN,NaN,NaN,Negative_Balanced_LengthMatched_PrimaryCoordinates_v1,Sampled from Negative Candidate Pool Clean v1 using 1:1 positive-negative balance with sequence-length-stratified sampling based on positive-set decile bins; non-primary patch/scaffold-only genes were excluded and replaced within the same length bins; random_state=42,NaN,one_uniprot_to_one_ensg,NaN,"(40.999, 210.0]",ENSG00000224420,ADM5,protein_coding,19,49688953.0,49690575.0,1.0
2,ENSG00000106113,CRHR2,corticotropin releasing hormone receptor 2,Q13324,CRFR2_HUMAN,MDAALLHSLLEANCSLALAEELLLDGWGPPLDPEGPYSYCNTTLDQIGTCWPRSAAGALVERPCPEYFNGVKYNTTRNAYRECLENGTWASKINYSQCEPILDDKQRKYDLHYRIALVVNYLGHCVSVAALVAAFLLFLALRSIRCLRNVIHWNLITTFILRNVMWFLLQLVDHEVHESNEVWCRCITTIFNYFVVTNFFWMFVEGCYLHTAIVMTYSTERLRKCLFLFIGWCIPFPIIVAWAIGKLYYENEQCWFGKEPGDLVDYIYQGPIILVLLINFVFLFNIVRILMTKLRASTTSETIQYRKAVKATLVLLPLLGITYMLFFVNPGEDDLSQIMFIYFNSFLQSFQGFFVSVFYCFFNGEVRSAVRKRWHRWQDHHSLRVPMARAMSIPTSPTRISFHSIKQTAAV,411.0,1,T2D_positive,positive,Category_A_T2D_positive_label,T2D_Protein_Classification_Balanced_PrimaryCoordinates_v1,0.526636,Tier_A_high_confidence_ge_0.50,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,NaN,NaN,primary_ENST_to_ENSG,one_uniprot_to_one_ensg,single_selected_sequence,NaN,ENSG00000106113,CRHR2,protein_coding,7,30651444.0,30700129.0,-1.0
3,ENSG00000165392,WRN,WRN RecQ like helicase,Q14191,WRN_HUMAN,MSEKKLETTAQQRKCPEWMNVQNKRCAVEERKACVRKSVFEDDLPFLEFTGSIVYSYDASDCSFLSEDISMSLSDGDVVGFDMEWPPLYNRGKLGKVALIQLCVSESKCYLFHVSSMSVFPQGLKMLLENKAVKKAGVGIEGDQWKLLRDFDIKLKNFVELTDVANKKLKCTETWSLNSLVKHLLGKQLLKDKSIRCSNWSKFPLTEDQKLYAATDAYAGFIIYRNLEILDDTVQRFAINKEEEILLSDMNKQLTSISEEVMDLAKHLPHAFSKLENPRRVSILLKDISENLYSLRRMIIGSTNIETE


Columns:
- gene_id
- gene_symbol
- gene_name
- uniprot_accession
- uniprot_entry_name
- sequence
- sequence_length_fasta
- label
- label_name
- dataset_role
- class_source
- classification_dataset_version
- association_score
- positive_tier
- label_source
- label_rule
- label_version
- negative_sample_version
- negative_sampling_rule
- mapping_route
- uniprot_to_ensg_status
- representative_sequence_rule
- length_bin
- ensembl_gene_id
- ensembl_gene_name
- gene_type
- chromosome_or_scaffold
- gene_start_bp
- gene_end_bp
- strand


In [4]:
# ============================================================
# REQUIRED COLUMNS CHECK
# ============================================================

required_cols = [
    "gene_id",
    "gene_symbol",
    "uniprot_accession",
    "sequence",
    "sequence_length_fasta",
    "label"
]

missing_cols = [col for col in required_cols if col not in df.columns]

print("Missing required columns:", missing_cols)

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")


# ============================================================
# LABEL DISTRIBUTION
# ============================================================

print("Label counts:")
print(df["label"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts(normalize=True))


# ============================================================
# MISSING VALUES
# ============================================================

print("Missing values in required columns:")
print(df[required_cols].isna().sum())

# gene_symbol is metadata only.
# If one symbol is missing, keep the row and fill UNKNOWN.
df["gene_symbol"] = df["gene_symbol"].fillna("UNKNOWN")


# ============================================================
# DUPLICATE CHECK
# ============================================================

print("Duplicate gene_id:", df["gene_id"].duplicated().sum())
print("Duplicate uniprot_accession:", df["uniprot_accession"].duplicated().sum())
print("Duplicate sequence:", df["sequence"].duplicated().sum())


# ============================================================
# LENGTH CHECK
# ============================================================

df["computed_sequence_length"] = df["sequence"].astype(str).str.len()

length_mismatch = df[df["computed_sequence_length"] != df["sequence_length_fasta"]]

print("Length mismatch rows:", len(length_mismatch))
display(df["computed_sequence_length"].describe())

if len(length_mismatch) > 0:
    display(length_mismatch[
        ["gene_id", "gene_symbol", "uniprot_accession", "sequence_length_fasta", "computed_sequence_length"]
    ].head())

Missing required columns: []
Label counts:
label
0    910
1    910
Name: count, dtype: int64

Label distribution:
label
0    0.5
1    0.5
Name: proportion, dtype: float64
Missing values in required columns:
gene_id                  0
gene_symbol              1
uniprot_accession        0
sequence                 0
sequence_length_fasta    0
label                    0
dtype: int64
Duplicate gene_id: 0
Duplicate uniprot_accession: 0
Duplicate sequence: 0
Length mismatch rows: 0


count     1820.000000
mean       767.783516
std       1015.463258
min         41.000000
25%        353.000000
50%        561.000000
75%        941.000000
max      34350.000000
Name: computed_sequence_length, dtype: float64

In [5]:
# ============================================================
# CLEAN SEQUENCES
# ============================================================

STANDARD_AAS = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AAS)

def find_non_standard_aas(seq):
    seq = str(seq).upper()
    return set(seq) - STANDARD_AA_SET

def clean_sequence(seq):
    """
    Keep only 20 standard amino acids.
    Non-standard residues such as U are removed for handcrafted AAC/DPC features.
    Original sequence is preserved in df["sequence"].
    """
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AA_SET])
    return seq


df["non_standard_aas"] = df["sequence"].apply(find_non_standard_aas)

print("Non-standard amino acid patterns:")
print(df["non_standard_aas"].value_counts())

rows_with_non_standard = df[df["non_standard_aas"].apply(lambda x: len(x) > 0)]
print("Rows with non-standard amino acids:", len(rows_with_non_standard))

if len(rows_with_non_standard) > 0:
    display(rows_with_non_standard[
        ["gene_id", "gene_symbol", "uniprot_accession", "label", "sequence_length_fasta", "non_standard_aas"]
    ])


df["sequence_clean"] = df["sequence"].apply(clean_sequence)
df["sequence_clean_length"] = df["sequence_clean"].str.len()

df["removed_non_standard_count"] = (
    df["sequence"].astype(str).str.len() - df["sequence_clean"].str.len()
)

print("Rows where non-standard residues were removed:")
display(df[df["removed_non_standard_count"] > 0][
    [
        "gene_id",
        "gene_symbol",
        "uniprot_accession",
        "label",
        "sequence_length_fasta",
        "sequence_clean_length",
        "removed_non_standard_count",
        "non_standard_aas"
    ]
])

Non-standard amino acid patterns:
non_standard_aas
{}     1819
{U}       1
Name: count, dtype: int64
Rows with non-standard amino acids: 1


,gene_id,gene_symbol,uniprot_accession,label,sequence_length_fasta,non_standard_aas
937,ENSG00000179918,SEPHS2,Q99611,0,448.0,{U}


Rows where non-standard residues were removed:


,gene_id,gene_symbol,uniprot_accession,label,sequence_length_fasta,sequence_clean_length,removed_non_standard_count,non_standard_aas
937,ENSG00000179918,SEPHS2,Q99611,0,448.0,447,1,{U}


In [6]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# 70 / 15 / 15
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts())

print("\nValidation label distribution:")
print(val_df["label"].value_counts())

print("\nTest label distribution:")
print(test_df["label"].value_counts())


# Save splits
train_df.to_csv(SPLIT_DIR / "train_protein_v1.csv", index=False)
val_df.to_csv(SPLIT_DIR / "val_protein_v1.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test_protein_v1.csv", index=False)

print("Split files saved.")

Train shape: (1274, 35)
Validation shape: (273, 35)
Test shape: (273, 35)

Train label distribution:
label
0    637
1    637
Name: count, dtype: int64

Validation label distribution:
label
1    137
0    136
Name: count, dtype: int64

Test label distribution:
label
0    137
1    136
Name: count, dtype: int64
Split files saved.


In [7]:
# ============================================================
# FEATURE EXTRACTION SETUP
# ============================================================

DIPEPTIDES = ["".join(pair) for pair in product(STANDARD_AAS, repeat=2)]

AA_GROUPS = {
    "hydrophobic": set("AVILMFWY"),
    "polar": set("STNQC"),
    "positive_charge": set("KRH"),
    "negative_charge": set("DE"),
    "special": set("GP")
}

print("Standard amino acids:", len(STANDARD_AAS))
print("Dipeptides:", len(DIPEPTIDES))


# ============================================================
# FEATURE GROUP 1: LENGTH
# ============================================================

def extract_length_feature(seq):
    seq = clean_sequence(seq)
    return {
        "protein_length": len(seq)
    }


# ============================================================
# FEATURE GROUP 2: AAC
# ============================================================

def extract_aac(seq):
    seq = clean_sequence(seq)
    length = len(seq)

    features = {}

    for aa in STANDARD_AAS:
        features[f"AAC_{aa}"] = seq.count(aa) / length if length > 0 else 0.0

    return features


# ============================================================
# FEATURE GROUP 3: DPC
# ============================================================

def extract_dpc(seq):
    seq = clean_sequence(seq)
    total_pairs = len(seq) - 1

    features = {f"DPC_{dp}": 0.0 for dp in DIPEPTIDES}

    if total_pairs <= 0:
        return features

    for i in range(total_pairs):
        pair = seq[i:i+2]

        if pair in DIPEPTIDES:
            features[f"DPC_{pair}"] += 1

    for dp in DIPEPTIDES:
        features[f"DPC_{dp}"] = features[f"DPC_{dp}"] / total_pairs

    return features


# ============================================================
# FEATURE GROUP 4: PHYSCHEM
# ============================================================

def extract_physchem(seq):
    seq = clean_sequence(seq)
    length = len(seq)

    if length == 0:
        return {
            "frac_hydrophobic": 0.0,
            "frac_polar": 0.0,
            "frac_positive_charge": 0.0,
            "frac_negative_charge": 0.0,
            "frac_special": 0.0,
            "charge_balance": 0.0
        }

    hydrophobic_count = sum(seq.count(aa) for aa in AA_GROUPS["hydrophobic"])
    polar_count = sum(seq.count(aa) for aa in AA_GROUPS["polar"])
    positive_count = sum(seq.count(aa) for aa in AA_GROUPS["positive_charge"])
    negative_count = sum(seq.count(aa) for aa in AA_GROUPS["negative_charge"])
    special_count = sum(seq.count(aa) for aa in AA_GROUPS["special"])

    return {
        "frac_hydrophobic": hydrophobic_count / length,
        "frac_polar": polar_count / length,
        "frac_positive_charge": positive_count / length,
        "frac_negative_charge": negative_count / length,
        "frac_special": special_count / length,
        "charge_balance": (positive_count - negative_count) / length
    }


# ============================================================
# COMBINE ALL FEATURES
# ============================================================

def extract_all_features(seq):
    features = {}

    features.update(extract_length_feature(seq))
    features.update(extract_aac(seq))
    features.update(extract_dpc(seq))
    features.update(extract_physchem(seq))

    return features


def build_feature_matrix(input_df, sequence_col="sequence_clean"):
    feature_dicts = input_df[sequence_col].apply(extract_all_features)

    feature_df = pd.DataFrame(
        feature_dicts.tolist(),
        index=input_df.index
    )

    return feature_df

Standard amino acids: 20
Dipeptides: 400


In [8]:
# ============================================================
# BUILD FULL FEATURE MATRICES
# ============================================================

X_train_full = build_feature_matrix(train_df, sequence_col="sequence_clean")
X_val_full = build_feature_matrix(val_df, sequence_col="sequence_clean")
X_test_full = build_feature_matrix(test_df, sequence_col="sequence_clean")

y_train = train_df["label"].astype(int)
y_val = val_df["label"].astype(int)
y_test = test_df["label"].astype(int)

print("X_train_full:", X_train_full.shape)
print("X_val_full:", X_val_full.shape)
print("X_test_full:", X_test_full.shape)

expected_features = 1 + 20 + 400 + 6
print("Expected features:", expected_features)
print("Actual features:", X_train_full.shape[1])

assert X_train_full.shape[1] == expected_features
assert X_val_full.shape[1] == expected_features
assert X_test_full.shape[1] == expected_features

assert list(X_train_full.columns) == list(X_val_full.columns)
assert list(X_train_full.columns) == list(X_test_full.columns)

print("Missing values X_train:", X_train_full.isna().sum().sum())
print("Missing values X_val:", X_val_full.isna().sum().sum())
print("Missing values X_test:", X_test_full.isna().sum().sum())

X_train_full: (1274, 427)
X_val_full: (273, 427)
X_test_full: (273, 427)
Expected features: 427
Actual features: 427
Missing values X_train: 0
Missing values X_val: 0
Missing values X_test: 0


In [9]:
# ============================================================
# AAC / DPC SUM CHECK
# ============================================================

aac_cols = [c for c in X_train_full.columns if c.startswith("AAC_")]
dpc_cols = [c for c in X_train_full.columns if c.startswith("DPC_")]
physchem_cols = [
    c for c in X_train_full.columns
    if c.startswith("frac_") or c == "charge_balance"
]
length_cols = ["protein_length"]

print("AAC cols:", len(aac_cols))
print("DPC cols:", len(dpc_cols))
print("Physchem cols:", len(physchem_cols))
print("Length cols:", len(length_cols))

aac_sum_train = X_train_full[aac_cols].sum(axis=1)
dpc_sum_train = X_train_full[dpc_cols].sum(axis=1)

print("\nAAC sum summary:")
display(aac_sum_train.describe())

print("\nDPC sum summary:")
display(dpc_sum_train.describe())

AAC cols: 20
DPC cols: 400
Physchem cols: 6
Length cols: 1

AAC sum summary:


count    1.274000e+03
mean     1.000000e+00
std      1.280716e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


DPC sum summary:


count    1.274000e+03
mean     1.000000e+00
std      1.946338e-15
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [10]:
# ============================================================
# SAVE FULL FEATURE MATRICES WITH METADATA
# ============================================================

def attach_metadata(split_df, X, y):
    output_df = X.copy()
    output_df.insert(0, "label", y.values)

    for col in [
        "sequence_clean_length",
        "sequence_length_fasta",
        "uniprot_accession",
        "gene_symbol",
        "gene_id"
    ]:
        if col in split_df.columns:
            output_df.insert(0, col, split_df[col].values)

    return output_df


train_full_out = attach_metadata(train_df, X_train_full, y_train)
val_full_out = attach_metadata(val_df, X_val_full, y_val)
test_full_out = attach_metadata(test_df, X_test_full, y_test)

train_full_out.to_csv(FEATURE_DIR / "full_features_train_v1.csv", index=False)
val_full_out.to_csv(FEATURE_DIR / "full_features_val_v1.csv", index=False)
test_full_out.to_csv(FEATURE_DIR / "full_features_test_v1.csv", index=False)

print("Full feature files saved.")

Full feature files saved.


In [11]:
# ============================================================
# FEATURE SUBSETS
# ============================================================

feature_sets = {
    "Length only": length_cols,
    "AAC only": aac_cols,
    "AAC + Physchem": aac_cols + physchem_cols,
    "AAC + DPC": aac_cols + dpc_cols,
    "Full": length_cols + aac_cols + dpc_cols + physchem_cols
}

for name, cols in feature_sets.items():
    print(name, ":", len(cols), "features")


# Main compact baseline feature set
main_feature_set_name = "AAC + Physchem"
main_cols = feature_sets[main_feature_set_name]

X_train = X_train_full[main_cols].copy()
X_val = X_val_full[main_cols].copy()
X_test = X_test_full[main_cols].copy()

print("\nMain compact baseline feature set:", main_feature_set_name)
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

assert X_train.shape[1] == 26

Length only : 1 features
AAC only : 20 features
AAC + Physchem : 26 features
AAC + DPC : 420 features
Full : 427 features

Main compact baseline feature set: AAC + Physchem
X_train: (1274, 26)
X_val: (273, 26)
X_test: (273, 26)


In [12]:
# ============================================================
# SAVE COMPACT FEATURE MATRICES
# ============================================================

train_compact_out = attach_metadata(train_df, X_train, y_train)
val_compact_out = attach_metadata(val_df, X_val, y_val)
test_compact_out = attach_metadata(test_df, X_test, y_test)

train_compact_out.to_csv(FEATURE_DIR / "compact_aac_physchem_train_v1.csv", index=False)
val_compact_out.to_csv(FEATURE_DIR / "compact_aac_physchem_val_v1.csv", index=False)
test_compact_out.to_csv(FEATURE_DIR / "compact_aac_physchem_test_v1.csv", index=False)

print("Compact feature files saved.")

Compact feature files saved.


In [13]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_positive_class_score(model, X):
    """
    Return continuous positive-class score for ROC-AUC and PR-AUC.
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


def evaluate_binary_classifier(model, X, y, model_name, dataset_name):
    """
    Evaluate binary classifier using project core metrics.
    """
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "model": model_name,
        "dataset": dataset_name,

        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),

        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [14]:
# ============================================================
# STRATIFIED 5-FOLD CV
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [15]:
# ============================================================
# MAIN MODEL PIPELINES
# Feature set: AAC + Physchem
# ============================================================

baseline_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}


if LIGHTGBM_AVAILABLE:
    baseline_models["LightGBM"] = Pipeline([
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=15,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])
else:
    baseline_models["Hist Gradient Boosting"] = Pipeline([
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            random_state=42
        ))
    ])

print("Main compact baseline models:")
for name in baseline_models:
    print("-", name)

Main compact baseline models:
- Logistic Regression
- SVM RBF
- Random Forest
- LightGBM


In [16]:
# ============================================================
# BASELINE CV BEFORE TUNING
# ============================================================

baseline_cv_results = []

for model_name, pipeline in baseline_models.items():
    print("=" * 80)
    print("Baseline CV:", model_name)

    cv_output = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    row = {
        "feature_set": main_feature_set_name,
        "model": model_name,

        "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
        "train_roc_auc_std": cv_output["train_roc_auc"].std(),

        "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

        "cv_f1_mean": cv_output["test_f1"].mean(),
        "cv_f1_std": cv_output["test_f1"].std(),

        "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
        "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

        "cv_mcc_mean": cv_output["test_mcc"].mean(),
        "cv_mcc_std": cv_output["test_mcc"].std(),

        "overfit_gap_train_minus_cv": (
            cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
        )
    }

    baseline_cv_results.append(row)

baseline_cv_df = pd.DataFrame(baseline_cv_results).sort_values(
    by="cv_roc_auc_mean",
    ascending=False
)

display(baseline_cv_df)

baseline_cv_df.to_csv(
    RESULT_DIR / "main_compact_baseline_cv_before_tuning_v1.csv",
    index=False
)

Baseline CV: Logistic Regression
Baseline CV: SVM RBF
Baseline CV: Random Forest
Baseline CV: LightGBM


,feature_set,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap_train_minus_cv
1,AAC + Physchem,SVM RBF,0.843441,0.006826,0.648895,0.039620,0.608031,0.028586,0.636128,0.042679,0.209560,0.049324,0.194546
2,AAC + Physchem,Random Forest,1.000000,0.000000,0.642360,0.047615,0.589073,0.038949,0.644658,0.046378,0.177514,0.073240,0.357640
0,AAC + Physchem,Logistic Regression,0.653019,0.012546,0.619656,0.043167,0.585933,0.025361,0.597658,0.030236,0.171701,0.068180,0.033363
3,AAC + Physchem,LightGBM,1.000000,0.000000,0.617308,0.052998,0.592746,0.042311,0.614421,0.047300,0.177758,0.079566,0.382692


In [17]:
# ============================================================
# PARAMETER GRIDS FOR COMPACT FEATURES
# ============================================================

param_grids = {
    "Logistic Regression": {
        "model__C": [0.001, 0.003, 0.01, 0.03, 0.1, 1],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    },

    "SVM RBF": {
        "model__C": [0.01, 0.1, 1, 10],
        "model__gamma": [0.0001, 0.001, 0.01, "scale"]
    },

    "Random Forest": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [3, 5, 8, 10],
        "model__min_samples_leaf": [5, 10, 20],
        "model__max_features": ["sqrt", 0.2, 0.5],
        "model__bootstrap": [True]
    }
}


if LIGHTGBM_AVAILABLE:
    param_grids["LightGBM"] = {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__num_leaves": [3, 7, 15],
        "model__max_depth": [2, 3, 5],
        "model__min_child_samples": [20, 50, 100],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
        "model__reg_alpha": [0, 0.1],
        "model__reg_lambda": [0.1, 1, 5]
    }
else:
    param_grids["Hist Gradient Boosting"] = {
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__max_iter": [100, 200, 300],
        "model__max_leaf_nodes": [7, 15, 31],
        "model__min_samples_leaf": [20, 50, 100]
    }

In [18]:
# ============================================================
# GRIDSEARCHCV
# ============================================================

grid_search_results = []
best_estimators = {}

for model_name, pipeline in baseline_models.items():
    print("=" * 80)
    print("GridSearchCV:", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train, y_train)

    best_estimators[model_name] = grid.best_estimator_

    best_idx = grid.best_index_

    result_row = {
        "feature_set": main_feature_set_name,
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": (
            grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_
        ),
        "best_params": grid.best_params_
    }

    grid_search_results.append(result_row)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", result_row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)

grid_results_df = pd.DataFrame(grid_search_results).sort_values(
    by="best_cv_roc_auc",
    ascending=False
)

display(grid_results_df)

grid_results_df.to_csv(
    RESULT_DIR / "main_compact_gridsearch_results_v1.csv",
    index=False
)

GridSearchCV: Logistic Regression
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best CV ROC-AUC: 0.6224283511067022
Best train ROC-AUC: 0.6515134620279792
Gap: 0.02908511092127697
Best params: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
GridSearchCV: SVM RBF
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best CV ROC-AUC: 0.6502242660735321
Best train ROC-AUC: 0.7234403014104445
Gap: 0.07321603533691234
Best params: {'model__C': 1, 'model__gamma': 0.01}
GridSearchCV: Random Forest
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best CV ROC-AUC: 0.6524553986607973
Best train ROC-AUC: 0.9948060705078673
Gap: 0.34235067184707
Best params: {'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 500}
GridSearchCV: LightGBM
Fitting 5 folds for each of 5832 candidates, totalling 29160 fits
Best CV ROC-AUC: 0.6453993970487941
Best train ROC-AUC: 0.890

,feature_set,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
2,AAC + Physchem,Random Forest,0.652455,0.994806,0.342351,"{'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 500}"
1,AAC + Physchem,SVM RBF,0.650224,0.723440,0.073216,"{'model__C': 1, 'model__gamma': 0.01}"
3,AAC + Physchem,LightGBM,0.645399,0.890131,0.244731,"{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.03, 'model__max_depth': 5, 'model__min_child_samples': 50, 'model__n_estimators': 200, 'model__num_leaves': 7, 'model__reg_alpha': 0.1, 'model__reg_lambda': 5, 'model__subsample': 0.8}"
0,AAC + Physchem,Logistic Regression,0.622428,0.651513,0.029085,"{'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"


In [19]:
# ============================================================
# EVALUATE TUNED INDIVIDUAL MODELS
# ============================================================

tuned_eval_records = []

for model_name, model in best_estimators.items():
    print("=" * 80)
    print("Evaluating tuned model:", model_name)

    train_metrics = evaluate_binary_classifier(
        model=model,
        X=X_train,
        y=y_train,
        model_name=model_name,
        dataset_name="train"
    )

    val_metrics = evaluate_binary_classifier(
        model=model,
        X=X_val,
        y=y_val,
        model_name=model_name,
        dataset_name="validation"
    )

    train_metrics["feature_set"] = main_feature_set_name
    val_metrics["feature_set"] = main_feature_set_name

    tuned_eval_records.extend([train_metrics, val_metrics])

tuned_eval_df = pd.DataFrame(tuned_eval_records)

display(tuned_eval_df)

tuned_eval_df.to_csv(
    RESULT_DIR / "main_compact_tuned_train_validation_metrics_v1.csv",
    index=False
)

validation_individual_comparison = tuned_eval_df[
    tuned_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(validation_individual_comparison)

Evaluating tuned model: Logistic Regression
Evaluating tuned model: SVM RBF
Evaluating tuned model: Random Forest
Evaluating tuned model: LightGBM


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,feature_set
0,Logistic Regression,train,0.600471,0.597264,0.616954,0.583987,0.606950,0.648780,0.628178,0.201051,372,265,244,393,AAC + Physchem
1,Logistic Regression,validation,0.564103,0.566176,0.562044,0.566176,0.564103,0.602458,0.581522,0.128220,77,59,60,77,AAC + Physchem
2,SVM RBF,train,0.648352,0.639175,0.681319,0.615385,0.659574,0.719530,0.695650,0.297350,392,245,203,434,AAC + Physchem
3,SVM RBF,validation,0.582418,0.585185,0.576642,0.588235,0.580882,0.623014,0.606311,0.164886,80,56,58,79,AAC + Physchem
4,Random Forest,train,0.954474,0.932735,0.979592,0.929356,0.955590,0.992599,0.992797,0.910097,592,45,13,624,AAC + Physchem
5,Random Forest,validation,0.622711,0.625000,0.620438,0.625000,0.622711,0.664609,0.661341,0.245438,85,51,52,85,AAC + Physchem
6,LightGBM,train,0.777080,0.775351,0.780220,0.773940,0.777778,0.870029,0.874892,0.554171,493,144,140,497,AAC + Physchem
7,LightGBM,validation,0.597070,0.601504,0.583942,0.610294,0.592593,0.629777,0.610860,0.194298,83,53,57,80,AAC + Physchem


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,feature_set
5,Random Forest,validation,0.622711,0.625000,0.620438,0.625000,0.622711,0.664609,0.661341,0.245438,85,51,52,85,AAC + Physchem
7,LightGBM,validation,0.597070,0.601504,0.583942,0.610294,0.592593,0.629777,0.610860,0.194298,83,53,57,80,AAC + Physchem
3,SVM RBF,validation,0.582418,0.585185,0.576642,0.588235,0.580882,0.623014,0.606311,0.164886,80,56,58,79,AAC + Physchem
1,Logistic Regression,validation,0.564103,0.566176,0.562044,0.566176,0.564103,0.602458,0.581522,0.128220,77,59,60,77,AAC + Physchem


In [20]:
# ============================================================
# SOFT VOTING ENSEMBLE
# ============================================================

voting_estimators = []

for model_name, estimator in best_estimators.items():
    short_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    voting_estimators.append((short_name, estimator))

print("Voting estimators:")
print([name for name, _ in voting_estimators])

soft_voting = VotingClassifier(
    estimators=voting_estimators,
    voting="soft",
    n_jobs=-1
)

print("Training Soft Voting...")
soft_voting.fit(X_train, y_train)

voting_train_metrics = evaluate_binary_classifier(
    model=soft_voting,
    X=X_train,
    y=y_train,
    model_name="Soft Voting",
    dataset_name="train"
)

voting_val_metrics = evaluate_binary_classifier(
    model=soft_voting,
    X=X_val,
    y=y_val,
    model_name="Soft Voting",
    dataset_name="validation"
)

voting_train_metrics["feature_set"] = main_feature_set_name
voting_val_metrics["feature_set"] = main_feature_set_name

display(pd.DataFrame([voting_train_metrics, voting_val_metrics]))

best_estimators["Soft Voting"] = soft_voting

Voting estimators:
['logistic_regression', 'svm_rbf', 'random_forest', 'lightgbm']
Training Soft Voting...


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,feature_set
0,Soft Voting,train,0.806907,0.804992,0.810047,0.803768,0.807512,0.879340,0.882239,0.613827,512,125,121,516,AAC + Physchem
1,Soft Voting,validation,0.600733,0.606061,0.583942,0.617647,0.594796,0.640887,0.625549,0.201697,84,52,57,80,AAC + Physchem


In [21]:
# ============================================================
# STACKING ENSEMBLE
# ============================================================

stacking_estimators = []

for model_name, estimator in best_estimators.items():
    if model_name == "Soft Voting":
        continue

    short_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    stacking_estimators.append((short_name, estimator))

print("Stacking base estimators:")
print([name for name, _ in stacking_estimators])

stacking = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

print("Training Stacking...")
stacking.fit(X_train, y_train)

stacking_train_metrics = evaluate_binary_classifier(
    model=stacking,
    X=X_train,
    y=y_train,
    model_name="Stacking",
    dataset_name="train"
)

stacking_val_metrics = evaluate_binary_classifier(
    model=stacking,
    X=X_val,
    y=y_val,
    model_name="Stacking",
    dataset_name="validation"
)

stacking_train_metrics["feature_set"] = main_feature_set_name
stacking_val_metrics["feature_set"] = main_feature_set_name

display(pd.DataFrame([stacking_train_metrics, stacking_val_metrics]))

best_estimators["Stacking"] = stacking

Stacking base estimators:
['logistic_regression', 'svm_rbf', 'random_forest', 'lightgbm']
Training Stacking...


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,feature_set
0,Stacking,train,0.826531,0.825000,0.828885,0.824176,0.826938,0.906151,0.907665,0.653068,525,112,109,528,AAC + Physchem
1,Stacking,validation,0.600733,0.607692,0.576642,0.625000,0.591760,0.643356,0.629580,0.201870,85,51,58,79,AAC + Physchem


In [22]:
# ============================================================
# COMBINE INDIVIDUAL + ENSEMBLE RESULTS
# ============================================================

ensemble_records = [
    voting_train_metrics,
    voting_val_metrics,
    stacking_train_metrics,
    stacking_val_metrics
]

all_eval_df = pd.concat(
    [
        tuned_eval_df,
        pd.DataFrame(ensemble_records)
    ],
    ignore_index=True
)

all_validation_results = all_eval_df[
    all_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(all_validation_results)

all_eval_df.to_csv(
    RESULT_DIR / "main_compact_all_train_validation_metrics_v1.csv",
    index=False
)

all_validation_results.to_csv(
    RESULT_DIR / "main_compact_validation_comparison_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,feature_set
5,Random Forest,validation,0.622711,0.625000,0.620438,0.625000,0.622711,0.664609,0.661341,0.245438,85,51,52,85,AAC + Physchem
11,Stacking,validation,0.600733,0.607692,0.576642,0.625000,0.591760,0.643356,0.629580,0.201870,85,51,58,79,AAC + Physchem
9,Soft Voting,validation,0.600733,0.606061,0.583942,0.617647,0.594796,0.640887,0.625549,0.201697,84,52,57,80,AAC + Physchem
7,LightGBM,validation,0.597070,0.601504,0.583942,0.610294,0.592593,0.629777,0.610860,0.194298,83,53,57,80,AAC + Physchem
3,SVM RBF,validation,0.582418,0.585185,0.576642,0.588235,0.580882,0.623014,0.606311,0.164886,80,56,58,79,AAC + Physchem
1,Logistic Regression,validation,0.564103,0.566176,0.562044,0.566176,0.564103,0.602458,0.581522,0.128220,77,59,60,77,AAC + Physchem


In [23]:
# ============================================================
# OVERFITTING TABLE
# ============================================================

overfit_table = all_eval_df.pivot_table(
    index="model",
    columns="dataset",
    values=["roc_auc", "f1", "mcc"]
)

display(overfit_table)

overfit_table.to_csv(
    RESULT_DIR / "main_compact_overfitting_table_v1.csv"
)

f1                  mcc              roc_auc  \
dataset                 train validation     train validation     train   
model                                                                     
LightGBM             0.777778   0.592593  0.554171   0.194298  0.870029   
Logistic Regression  0.606950   0.564103  0.201051   0.128220  0.648780   
Random Forest        0.955590   0.622711  0.910097   0.245438  0.992599   
SVM RBF              0.659574   0.580882  0.297350   0.164886  0.719530   
Soft Voting          0.807512   0.594796  0.613827   0.201697  0.879340   
Stacking             0.826938   0.591760  0.653068   0.201870  0.906151   

                                
dataset             validation  
model                           
LightGBM              0.629777  
Logistic Regression   0.602458  
Random Forest         0.664609  
SVM RBF               0.623014  
Soft Voting           0.640887  
Stacking              0.643356

In [24]:
# ============================================================
# SELECT FINAL MODEL BASED ON VALIDATION
# ============================================================

best_validation_row = all_validation_results.iloc[0]
final_model_name = best_validation_row["model"]
final_model = best_estimators[final_model_name]

print("Final selected compact baseline model:", final_model_name)
display(best_validation_row)

pd.DataFrame([best_validation_row]).to_csv(
    RESULT_DIR / "main_compact_final_model_validation_summary_v1.csv",
    index=False
)

Final selected compact baseline model: Random Forest


model                  Random Forest
dataset                   validation
accuracy                    0.622711
precision                      0.625
recall_sensitivity          0.620438
specificity                    0.625
f1                          0.622711
roc_auc                     0.664609
pr_auc                      0.661341
mcc                         0.245438
tn                                85
fp                                51
fn                                52
tp                                85
feature_set           AAC + Physchem
Name: 5, dtype: object

In [35]:
# ============================================================
# FINAL TEST EVALUATION
# Test set used only after final model selection.
# ============================================================

final_test_metrics = evaluate_binary_classifier(
    model=final_model,
    X=X_test,
    y=y_test,
    model_name=final_model_name,
    dataset_name="test"
)

final_test_metrics["feature_set"] = main_feature_set_name

final_test_metrics_df = pd.DataFrame([final_test_metrics])

display(final_test_metrics_df)

final_test_metrics_df.to_csv(
    RESULT_DIR / "main_compact_final_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,feature_set
0,Random Forest,test,0.52381,0.520548,0.558824,0.489051,0.539007,0.551954,0.555038,0.047991,67,70,60,76,AAC + Physchem


In [25]:
# ============================================================
# FINAL TEST CONFUSION MATRIX
# ============================================================

y_test_pred = final_model.predict(X_test)

cm = confusion_matrix(y_test, y_test_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_0_Background", "Actual_1_T2D"],
    columns=["Pred_0_Background", "Pred_1_T2D"]
)

display(cm_df)

cm_df.to_csv(
    RESULT_DIR / "main_compact_final_test_confusion_matrix_v1.csv"
)

,Pred_0_Background,Pred_1_T2D
Actual_0_Background,67,70
Actual_1_T2D,60,76


In [26]:
# ============================================================
# SAVE FINAL TEST PREDICTIONS
# ============================================================

y_test_score = get_positive_class_score(final_model, X_test)
y_test_pred = final_model.predict(X_test)

test_predictions_df = pd.DataFrame({
    "true_label": y_test.values,
    "pred_label": y_test_pred,
    "pred_score_t2d_associated": y_test_score
})

metadata_cols = [
    col for col in [
        "gene_id",
        "gene_symbol",
        "uniprot_accession",
        "sequence_length_fasta",
        "sequence_clean_length"
    ]
    if col in test_df.columns
]

if len(metadata_cols) > 0:
    metadata_df = test_df[metadata_cols].reset_index(drop=True)

    test_predictions_df = pd.concat(
        [
            metadata_df,
            test_predictions_df.reset_index(drop=True)
        ],
        axis=1
    )

display(test_predictions_df.head())

test_predictions_df.to_csv(
    RESULT_DIR / "main_compact_final_test_predictions_v1.csv",
    index=False
)

,gene_id,gene_symbol,uniprot_accession,sequence_length_fasta,sequence_clean_length,true_label,pred_label,pred_score_t2d_associated
0,ENSG00000177542,SLC25A22,Q9H936,323.0,323,0,1,0.601578
1,ENSG00000123080,CDKN2C,P42773,168.0,168,1,0,0.460169
2,ENSG00000185262,UBALD2,Q8IYN6,164.0,164,0,1,0.554924
3,ENSG00000092148,HECTD1,Q9ULT8,2610.0,2610,0,1,0.555686
4,ENSG00000139364,TMEM132B,Q14DG7,1078.0,1078,1,1,0.539694


In [27]:
# ============================================================
# DPC ABLATION EXPERIMENT
# Purpose:
#   Test whether DPC improves generalization or mainly increases overfitting.
#
# Models:
#   Logistic Regression
#   Random Forest
# ============================================================

ablation_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=0.001,
            penalty="l2",
            solver="lbfgs",
            max_iter=5000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            max_features="sqrt",
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

In [28]:
# ============================================================
# RUN ABLATION CV ON TRAIN SET
# ============================================================

ablation_records = []

for feature_set_name, cols in feature_sets.items():
    Xtr_sub = X_train_full[cols].copy()

    for model_name, model in ablation_models.items():
        print("=" * 80)
        print(feature_set_name, "|", model_name)

        cv_output = cross_validate(
            estimator=model,
            X=Xtr_sub,
            y=y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=True
        )

        row = {
            "feature_set": feature_set_name,
            "n_features": len(cols),
            "model": model_name,

            "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
            "train_roc_auc_std": cv_output["train_roc_auc"].std(),

            "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
            "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

            "cv_f1_mean": cv_output["test_f1"].mean(),
            "cv_f1_std": cv_output["test_f1"].std(),

            "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
            "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

            "cv_mcc_mean": cv_output["test_mcc"].mean(),
            "cv_mcc_std": cv_output["test_mcc"].std(),

            "overfit_gap": (
                cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
            )
        }

        ablation_records.append(row)

ablation_cv_df = pd.DataFrame(ablation_records).sort_values(
    by=["cv_roc_auc_mean", "cv_mcc_mean"],
    ascending=False
)

display(ablation_cv_df)

ablation_cv_df.to_csv(
    RESULT_DIR / "dpc_ablation_cv_results_v1.csv",
    index=False
)

Length only | Logistic Regression
Length only | Random Forest
AAC only | Logistic Regression
AAC only | Random Forest
AAC + Physchem | Logistic Regression
AAC + Physchem | Random Forest
AAC + DPC | Logistic Regression
AAC + DPC | Random Forest
Full | Logistic Regression
Full | Random Forest


,feature_set,n_features,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap
5,AAC + Physchem,26,Random Forest,0.944876,0.004826,0.650436,0.048805,0.616848,0.039815,0.649417,0.041039,0.218550,0.089961,0.294440
9,Full,427,Random Forest,0.998864,0.000774,0.649915,0.034630,0.615495,0.020602,0.637021,0.033866,0.220113,0.040098,0.348949
3,AAC only,20,Random Forest,0.940967,0.003473,0.643354,0.052381,0.602763,0.039046,0.640019,0.038543,0.190167,0.085285,0.297613
7,AAC + DPC,420,Random Forest,0.998957,0.000527,0.641280,0.043342,0.590815,0.043137,0.629810,0.039132,0.184059,0.083742,0.357678
6,AAC + DPC,420,Logistic Regression,0.793092,0.005396,0.631112,0.033138,0.600989,0.030761,0.610982,0.027475,0.190317,0.072047,0.161980
8,Full,427,Logistic Regression,0.792608,0.005393,0.630829,0.034117,0.594712,0.027060,0.610408,0.028775,0.177747,0.064565,0.161780
2,AAC only,20,Logistic Regression,0.639785,0.009858,0.619368,0.040023,0.583935,0.028966,0.597005,0.036203,0.163757,0.045736,0.020417
4,AAC + Physchem,26,Logistic Regression,0.636755,0.009960,0.616484,0.038391,0.590235,0.032137,0.593881,0.036351,0.176457,0.058138,0.020271
1,Length only,1,Random Forest,0.710575,0.004439,0.510237,0.003517,0.516447,0.028338,0.506585,0.009921,0.026943,0.016459,0.200338
0,Length only,1,Logistic Regression,0.503161,0.006437,0.491936,0.023136,0.483975,0.127761,0.501221,0.033437,-0.006177,0.052782,0.011225


In [29]:
# ============================================================
# DPC-FOCUSED COMPARISON
# ============================================================

dpc_comparison_df = ablation_cv_df[
    ablation_cv_df["feature_set"].isin([
        "AAC only",
        "AAC + Physchem",
        "AAC + DPC",
        "Full"
    ])
].copy()

dpc_comparison_df = dpc_comparison_df.sort_values(
    by=["model", "cv_roc_auc_mean"],
    ascending=[True, False]
)

display(dpc_comparison_df)

dpc_comparison_df.to_csv(
    RESULT_DIR / "dpc_focused_ablation_comparison_v1.csv",
    index=False
)

,feature_set,n_features,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap
6,AAC + DPC,420,Logistic Regression,0.793092,0.005396,0.631112,0.033138,0.600989,0.030761,0.610982,0.027475,0.190317,0.072047,0.161980
8,Full,427,Logistic Regression,0.792608,0.005393,0.630829,0.034117,0.594712,0.027060,0.610408,0.028775,0.177747,0.064565,0.161780
2,AAC only,20,Logistic Regression,0.639785,0.009858,0.619368,0.040023,0.583935,0.028966,0.597005,0.036203,0.163757,0.045736,0.020417
4,AAC + Physchem,26,Logistic Regression,0.636755,0.009960,0.616484,0.038391,0.590235,0.032137,0.593881,0.036351,0.176457,0.058138,0.020271
5,AAC + Physchem,26,Random Forest,0.944876,0.004826,0.650436,0.048805,0.616848,0.039815,0.649417,0.041039,0.218550,0.089961,0.294440
9,Full,427,Random Forest,0.998864,0.000774,0.649915,0.034630,0.615495,0.020602,0.637021,0.033866,0.220113,0.040098,0.348949
3,AAC only,20,Random Forest,0.940967,0.003473,0.643354,0.052381,0.602763,0.039046,0.640019,0.038543,0.190167,0.085285,0.297613
7,AAC + DPC,420,Random Forest,0.998957,0.000527,0.641280,0.043342,0.590815,0.043137,0.629810,0.039132,0.184059,0.083742,0.357678


In [30]:
# ============================================================
# SIMPLE TEXT CONCLUSION FOR DPC
# ============================================================

def get_ablation_row(feature_set, model_name):
    row = ablation_cv_df[
        (ablation_cv_df["feature_set"] == feature_set) &
        (ablation_cv_df["model"] == model_name)
    ]
    if len(row) == 0:
        return None
    return row.iloc[0]


for model_name in ["Logistic Regression", "Random Forest"]:
    print("=" * 80)
    print("Model:", model_name)

    aac = get_ablation_row("AAC only", model_name)
    aac_phys = get_ablation_row("AAC + Physchem", model_name)
    aac_dpc = get_ablation_row("AAC + DPC", model_name)
    full = get_ablation_row("Full", model_name)

    for label, row in [
        ("AAC only", aac),
        ("AAC + Physchem", aac_phys),
        ("AAC + DPC", aac_dpc),
        ("Full", full)
    ]:
        if row is not None:
            print(
                f"{label}: "
                f"CV ROC-AUC={row['cv_roc_auc_mean']:.4f}, "
                f"MCC={row['cv_mcc_mean']:.4f}, "
                f"Gap={row['overfit_gap']:.4f}, "
                f"n_features={int(row['n_features'])}"
            )

Model: Logistic Regression
AAC only: CV ROC-AUC=0.6194, MCC=0.1638, Gap=0.0204, n_features=20
AAC + Physchem: CV ROC-AUC=0.6165, MCC=0.1765, Gap=0.0203, n_features=26
AAC + DPC: CV ROC-AUC=0.6311, MCC=0.1903, Gap=0.1620, n_features=420
Full: CV ROC-AUC=0.6308, MCC=0.1777, Gap=0.1618, n_features=427
Model: Random Forest
AAC only: CV ROC-AUC=0.6434, MCC=0.1902, Gap=0.2976, n_features=20
AAC + Physchem: CV ROC-AUC=0.6504, MCC=0.2185, Gap=0.2944, n_features=26
AAC + DPC: CV ROC-AUC=0.6413, MCC=0.1841, Gap=0.3577, n_features=420
Full: CV ROC-AUC=0.6499, MCC=0.2201, Gap=0.3489, n_features=427


In [31]:
# ============================================================
# RANDOM FOREST FEATURE IMPORTANCE
# Compact AAC + Physchem
# ============================================================

if "Random Forest" in best_estimators:
    rf_pipeline = best_estimators["Random Forest"]
    rf_model = rf_pipeline.named_steps["model"]

    rf_importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf_model.feature_importances_
    }).sort_values("importance", ascending=False)

    display(rf_importance_df)

    rf_importance_df.to_csv(
        RESULT_DIR / "compact_random_forest_feature_importance_v1.csv",
        index=False
    )

,feature,importance
9,AAC_L,0.056767
11,AAC_N,0.050333
3,AAC_E,0.049824
22,frac_positive_charge,0.048063
19,AAC_Y,0.046741
8,AAC_K,0.045372
10,AAC_M,0.043559
6,AAC_H,0.042285
7,AAC_I,0.040726
5,AAC_G,0.037428


In [32]:
# ============================================================
# LOGISTIC REGRESSION COEFFICIENTS
# Compact AAC + Physchem
# ============================================================

if "Logistic Regression" in best_estimators:
    logreg_pipeline = best_estimators["Logistic Regression"]
    logreg_model = logreg_pipeline.named_steps["model"]

    logreg_coef_df = pd.DataFrame({
        "feature": X_train.columns,
        "coefficient": logreg_model.coef_[0]
    })

    logreg_coef_df["abs_coefficient"] = logreg_coef_df["coefficient"].abs()

    logreg_coef_df = logreg_coef_df.sort_values(
        by="abs_coefficient",
        ascending=False
    )

    display(logreg_coef_df)

    logreg_coef_df.to_csv(
        RESULT_DIR / "compact_logistic_regression_coefficients_v1.csv",
        index=False
    )

,feature,coefficient,abs_coefficient
9,AAC_L,-0.158238,0.158238
11,AAC_N,0.144499,0.144499
10,AAC_M,0.119848,0.119848
3,AAC_E,-0.116456,0.116456
2,AAC_D,0.115352,0.115352
0,AAC_A,0.101627,0.101627
19,AAC_Y,0.101045,0.101045
7,AAC_I,0.072725,0.072725
6,AAC_H,-0.064792,0.064792
4,AAC_F,-0.063079,0.063079


In [33]:
# ============================================================
# SAVE ALL TRAINED MODELS
# ============================================================

for model_name, model in best_estimators.items():
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_path = MODEL_DIR / f"compact_{safe_name}_best_estimator_v1.pkl"
    joblib.dump(model, model_path)

    print("Saved:", model_path)

final_model_path = MODEL_DIR / "compact_final_selected_model_v1.pkl"
joblib.dump(final_model, final_model_path)

print("Final model saved:", final_model_path)

Saved: D:\khoane\MAHE\Project\model\phase1_protein_baseline\models\compact_logistic_regression_best_estimator_v1.pkl
Saved: D:\khoane\MAHE\Project\model\phase1_protein_baseline\models\compact_svm_rbf_best_estimator_v1.pkl
Saved: D:\khoane\MAHE\Project\model\phase1_protein_baseline\models\compact_random_forest_best_estimator_v1.pkl
Saved: D:\khoane\MAHE\Project\model\phase1_protein_baseline\models\compact_lightgbm_best_estimator_v1.pkl
Saved: D:\khoane\MAHE\Project\model\phase1_protein_baseline\models\compact_soft_voting_best_estimator_v1.pkl
Saved: D:\khoane\MAHE\Project\model\phase1_protein_baseline\models\compact_stacking_best_estimator_v1.pkl
Final model saved: D:\khoane\MAHE\Project\model\phase1_protein_baseline\models\compact_final_selected_model_v1.pkl


In [36]:
# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

summary_tables = {
    "baseline_cv_before_tuning": baseline_cv_df,
    "gridsearch_results": grid_results_df,
    "validation_comparison": all_validation_results,
    "final_test_metrics": final_test_metrics_df,
    "dpc_ablation": ablation_cv_df
}

print("=" * 80)
print("PHASE 1.1 FINAL HANDCRAFTED BASELINE COMPLETE")
print("=" * 80)

print("\nMain feature set:", main_feature_set_name)
print("Number of compact features:", len(main_cols))

print("\nFinal selected model:")
print(final_model_name)

print("\nFinal test metrics:")
display(final_test_metrics_df)

print("\nDPC ablation top rows:")
display(ablation_cv_df.head(10))

print("\nResults saved to:", RESULT_DIR)
print("Models saved to:", MODEL_DIR)

PHASE 1.1 FINAL HANDCRAFTED BASELINE COMPLETE

Main feature set: AAC + Physchem
Number of compact features: 26

Final selected model:
Random Forest

Final test metrics:


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,feature_set
0,Random Forest,test,0.52381,0.520548,0.558824,0.489051,0.539007,0.551954,0.555038,0.047991,67,70,60,76,AAC + Physchem



DPC ablation top rows:


,feature_set,n_features,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap
5,AAC + Physchem,26,Random Forest,0.944876,0.004826,0.650436,0.048805,0.616848,0.039815,0.649417,0.041039,0.218550,0.089961,0.294440
9,Full,427,Random Forest,0.998864,0.000774,0.649915,0.034630,0.615495,0.020602,0.637021,0.033866,0.220113,0.040098,0.348949
3,AAC only,20,Random Forest,0.940967,0.003473,0.643354,0.052381,0.602763,0.039046,0.640019,0.038543,0.190167,0.085285,0.297613
7,AAC + DPC,420,Random Forest,0.998957,0.000527,0.641280,0.043342,0.590815,0.043137,0.629810,0.039132,0.184059,0.083742,0.357678
6,AAC + DPC,420,Logistic Regression,0.793092,0.005396,0.631112,0.033138,0.600989,0.030761,0.610982,0.027475,0.190317,0.072047,0.161980
8,Full,427,Logistic Regression,0.792608,0.005393,0.630829,0.034117,0.594712,0.027060,0.610408,0.028775,0.177747,0.064565,0.161780
2,AAC only,20,Logistic Regression,0.639785,0.009858,0.619368,0.040023,0.583935,0.028966,0.597005,0.036203,0.163757,0.045736,0.020417
4,AAC + Physchem,26,Logistic Regression,0.636755,0.009960,0.616484,0.038391,0.590235,0.032137,0.593881,0.036351,0.176457,0.058138,0.020271
1,Length only,1,Random Forest,0.710575,0.004439,0.510237,0.003517,0.516447,0.028338,0.506585,0.009921,0.026943,0.016459,0.200338
0,Length only,1,Logistic Regression,0.503161,0.006437,0.491936,0.023136,0.483975,0.127761,0.501221,0.033437,-0.006177,0.052782,0.011225



Results saved to: D:\khoane\MAHE\Project\model\phase1_protein_baseline\results
Models saved to: D:\khoane\MAHE\Project\model\phase1_protein_baseline\models


In [2]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import transformers
print(transformers.__version__)

2.2.0+cpu
CUDA available: False
4.56.2
